# 05. 연구 설계 보강과 통계적 검증

이 노트북은 PPT에서 부족했던 연구 설계 레이어를 보완합니다.

목표는 세 가지입니다.

1. 선행조사·선행논의 근거를 정리한다.
2. KGSS 성공 요인 분석에 신뢰구간과 검정 결과를 추가한다.
3. 가계금융복지조사 보조 분석에 관련성 검정과 효과크기를 추가한다.

주의: 이 노트북의 검정은 발표용 보강 장치입니다. KGSS와 가계금융복지조사의 복합표본설계를 완전히 반영한 설계기반 추론이 아니므로, p-value보다 방향·효과크기·해석의 일관성을 우선합니다.

## 1. 라이브러리와 경로

In [ ]:
from pathlib import Path
import math
import os

os.environ.setdefault('MPLCONFIGDIR', str(Path('/tmp') / 'matplotlib'))

import numpy as np
import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
from matplotlib import font_manager

In [ ]:
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == 'notebooks' else cwd
raw_dir = project_root / 'data' / 'raw'
table_dir = project_root / 'outputs' / 'tables'
figure_dir = project_root / 'outputs' / 'figures'
table_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

font_candidates = ['AppleGothic', 'Malgun Gothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR', 'Noto Sans KR']
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
selected_font = next((font for font in font_candidates if font in installed_fonts), 'DejaVu Sans')
plt.rcParams['font.family'] = selected_font
plt.rcParams['axes.unicode_minus'] = False

print('프로젝트 루트:', project_root)
print('선택된 한글 폰트:', selected_font)

## 2. 선행조사·선행논의 근거 정리

이 표는 PPT의 문제 배경, 데이터 선택, 분석 방법 슬라이드에 바로 사용할 수 있는 근거 목록입니다.

In [ ]:
research_context_sources = pd.DataFrame([
    {
        '구분': '메인 데이터 근거',
        '자료/문헌': '한국종합사회조사(KGSS), KOSSDA',
        '핵심 내용': '한국 사회의 태도·인식 변화를 반복적으로 관찰할 수 있는 사회조사 자료. 성공 요인 문항이 포함되어 본 프로젝트의 메인 자료로 적합함.',
        '우리 분석에서의 역할': '성공을 무엇으로 설명하는지 직접 측정하는 메인 데이터',
        'PPT 사용 위치': '데이터 선택과 분석 슬라이드',
        '주의점': '응답 인식 자료이므로 실제 성공 원인을 직접 측정하는 자료가 아님.',
        'URL': 'https://kossda.snu.ac.kr/',
    },
    {
        '구분': '국제조사 맥락',
        '자료/문헌': 'International Social Survey Programme(ISSP)',
        '핵심 내용': '사회불평등 등 주요 사회과학 주제를 국가 간 비교 가능하게 반복 조사하는 국제조사 프로그램.',
        '우리 분석에서의 역할': 'KGSS 성공 요인 문항을 사회불평등·사회이동성 인식 문항의 국제조사 맥락에서 설명',
        'PPT 사용 위치': '문제 배경 또는 데이터 선택 슬라이드',
        '주의점': '이번 분석은 KGSS 국내 자료 중심이며 국제비교 분석은 하지 않음.',
        'URL': 'https://issp.org/survey-topics/',
    },
    {
        '구분': '외부 조사 결과',
        '자료/문헌': '통계청 사회조사',
        '핵심 내용': '계층이동 가능성 등 사회 인식 항목을 포함해 한국인의 사회이동 기대를 보여주는 공식 조사.',
        '우리 분석에서의 역할': '성공 인식 문제가 개인 감상이 아니라 사회이동성 인식과 연결된 공적 조사 주제임을 보여줌',
        'PPT 사용 위치': '문제 배경 슬라이드',
        '주의점': 'KGSS 성공 요인 문항과 동일한 질문은 아니므로 직접 합산하거나 비교하지 않음.',
        'URL': 'https://kostat.go.kr/board.es?bid=219&mid=a10301010000',
    },
    {
        '구분': '선행 논의',
        '자료/문헌': 'OECD, A Broken Social Elevator: How to Promote Social Mobility',
        '핵심 내용': '사회이동성 약화와 부모 배경·불평등이 기회 구조에 미치는 영향을 다룬 국제 보고서.',
        '우리 분석에서의 역할': '성공을 노력만으로 설명하기 어려운 구조적 배경 논의 제시',
        'PPT 사용 위치': '선행논의/문제 배경 슬라이드',
        '주의점': 'OECD 논의는 배경 근거이며 KGSS 결과의 인과 설명으로 사용하지 않음.',
        'URL': 'https://doi.org/10.1787/9789264301085-en',
    },
    {
        '구분': '보조 경제 데이터 근거',
        '자료/문헌': '통계청 MDIS 가계금융복지조사 2025 가구마스터',
        '핵심 내용': '가구 단위 소득, 자산, 부채, 순자산, 주거 관련 변수를 포함한 마이크로데이터.',
        '우리 분석에서의 역할': '현재 소득과 축적 자산의 관계를 보여주는 경제적 배경 자료',
        'PPT 사용 위치': '보조 경제 분석 슬라이드',
        '주의점': '성공 인식의 원인을 직접 설명하는 자료가 아니며 경제적 배경으로만 사용.',
        'URL': 'https://mdis.kostat.go.kr/dwnlSvc/ofrSurvSearch.do?curMenuNo=UI_POR_P9240',
    },
])

sources_path = table_dir / '05_research_context_sources.csv'
research_context_sources.to_csv(sources_path, index=False, encoding='utf-8-sig')
print('저장:', sources_path)
display(research_context_sources)

## 3. 통계 함수

외부 통계 패키지 없이 재현 가능하도록 필요한 검정 함수를 직접 정의합니다.

- Wilson 신뢰구간: 가중 비율의 불확실성 확인
- Kish effective n: 가중치가 있는 비율의 보수적 표본크기 근사
- 카이제곱 독립성 검정: 범주형 변수 간 관련성 확인
- Cramer's V: 표본크기에 덜 민감한 효과크기
- Cochran-Armitage trend test: 연도 순서에 따른 이분 응답 추세 확인

In [ ]:
def kish_effective_n(weights):
    weights = np.asarray(weights, dtype=float)
    weights = weights[np.isfinite(weights) & (weights > 0)]
    if len(weights) == 0:
        return np.nan
    return weights.sum() ** 2 / np.square(weights).sum()


def wilson_ci_from_count(successes, n, confidence=0.95):
    if n <= 0 or not np.isfinite(n):
        return np.nan, np.nan
    # 95% 기준. PPT 보강 목적이므로 confidence 인자는 0.95만 사용한다.
    z = 1.959963984540054
    p = successes / n
    denominator = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denominator
    half_width = z * math.sqrt((p*(1-p) + z**2/(4*n)) / n) / denominator
    return max(0, center - half_width), min(1, center + half_width)


def weighted_binary_summary(data, indicator_col, weight_col):
    valid = data[[indicator_col, weight_col]].dropna().copy()
    weights = valid[weight_col].astype(float)
    y = valid[indicator_col].astype(float)
    weighted_p = (weights * y).sum() / weights.sum()
    n_eff = kish_effective_n(weights)
    ci_low, ci_high = wilson_ci_from_count(weighted_p * n_eff, n_eff)
    return {
        'weighted_pct': weighted_p * 100,
        'ci_low_pct': ci_low * 100,
        'ci_high_pct': ci_high * 100,
        'valid_n': int(len(valid)),
        'effective_n': n_eff,
    }


def gammainc_lower_regularized(a, x, eps=1e-12, max_iter=1000):
    if x <= 0:
        return 0.0
    if x < a + 1:
        ap = a
        summation = 1.0 / a
        delta = summation
        for _ in range(max_iter):
            ap += 1
            delta *= x / ap
            summation += delta
            if abs(delta) < abs(summation) * eps:
                break
        return summation * math.exp(-x + a * math.log(x) - math.lgamma(a))
    return 1.0 - gammainc_upper_regularized(a, x, eps=eps, max_iter=max_iter)


def gammainc_upper_regularized(a, x, eps=1e-12, max_iter=1000):
    if x <= 0:
        return 1.0
    if x < a + 1:
        return 1.0 - gammainc_lower_regularized(a, x, eps=eps, max_iter=max_iter)
    tiny = 1e-300
    b = x + 1.0 - a
    c = 1.0 / tiny
    d = 1.0 / max(b, tiny)
    h = d
    for i in range(1, max_iter + 1):
        an = -i * (i - a)
        b += 2.0
        d = an * d + b
        if abs(d) < tiny:
            d = tiny
        c = b + an / c
        if abs(c) < tiny:
            c = tiny
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < eps:
            break
    return math.exp(-x + a * math.log(x) - math.lgamma(a)) * h


def chi_square_sf(x, df):
    if df <= 0:
        return np.nan
    return min(1.0, max(0.0, gammainc_upper_regularized(df / 2, x / 2)))


def chi_square_independence(table):
    observed = np.asarray(table, dtype=float)
    total = observed.sum()
    row_sum = observed.sum(axis=1, keepdims=True)
    col_sum = observed.sum(axis=0, keepdims=True)
    expected = row_sum @ col_sum / total
    with np.errstate(divide='ignore', invalid='ignore'):
        chi2 = np.nansum((observed - expected) ** 2 / expected)
    df = (observed.shape[0] - 1) * (observed.shape[1] - 1)
    p_value = chi_square_sf(chi2, df)
    min_dim = min(observed.shape[0] - 1, observed.shape[1] - 1)
    cramers_v = math.sqrt(chi2 / (total * min_dim)) if total > 0 and min_dim > 0 else np.nan
    return chi2, df, p_value, cramers_v


def cochran_armitage_trend_test(successes, totals, scores):
    successes = np.asarray(successes, dtype=float)
    totals = np.asarray(totals, dtype=float)
    scores = np.asarray(scores, dtype=float)
    n = totals.sum()
    x = successes.sum()
    if n <= 1 or x <= 0 or x >= n:
        return np.nan, np.nan, np.nan
    score_bar = np.sum(totals * scores) / n
    numerator = np.sum(scores * (successes - totals * x / n))
    denominator = math.sqrt((x * (n - x) / (n * (n - 1))) * np.sum(totals * (scores - score_bar) ** 2))
    z = numerator / denominator
    chi2 = z ** 2
    p_value = math.erfc(abs(z) / math.sqrt(2))
    return z, chi2, p_value


def weighted_corr(x, y, w):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(w) & (w > 0)
    x, y, w = x[mask], y[mask], w[mask]
    mx = np.average(x, weights=w)
    my = np.average(y, weights=w)
    cov = np.average((x - mx) * (y - my), weights=w)
    vx = np.average((x - mx) ** 2, weights=w)
    vy = np.average((y - my) ** 2, weights=w)
    return cov / math.sqrt(vx * vy)


def format_p_value(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '<0.001'
    return f'{p:.3f}'

print('통계 함수 준비 완료')

## 4. KGSS: 성공 요인별 신뢰구간

In [ ]:
kgss_path = raw_dir / 'kor_data_CUM0074_V2.sav'
df, meta = pyreadstat.read_sav(kgss_path)

success_labels = {
    'SUCDEFRT': '열심히 일',
    'SUCDWLTH': '부유한 집안',
    'SUCDPAED': '부모 교육',
    'SUCDKNOW': '좋은 사람을 아는 것',
}
success_variables = list(success_labels.keys())
success_years = [2009, 2014, 2021, 2023, 2025]
important_values = [1, 2, 3]
valid_values = [1, 2, 3, 4, 5]

required = ['YEAR', 'AGE', 'FINALWT', *success_variables]
missing = [col for col in required if col not in df.columns]
if missing:
    raise KeyError(f'KGSS 필수 변수가 없습니다: {missing}')

kgss = df[required].copy()
print('KGSS shape:', kgss.shape)

In [ ]:
ci_rows = []
for year in success_years:
    year_df = kgss[kgss['YEAR'].eq(year)].copy()
    for variable, label in success_labels.items():
        valid = year_df[year_df[variable].isin(valid_values) & year_df['FINALWT'].notna()].copy()
        valid['important'] = valid[variable].isin(important_values).astype(int)
        summary = weighted_binary_summary(valid, 'important', 'FINALWT')
        ci_rows.append({
            'YEAR': year,
            'variable': variable,
            'label': label,
            **summary,
        })

kgss_success_ci_by_year = pd.DataFrame(ci_rows)
ci_path = table_dir / '05_kgss_success_ci_by_year.csv'
kgss_success_ci_by_year.to_csv(ci_path, index=False, encoding='utf-8-sig')
print('저장:', ci_path)
display(kgss_success_ci_by_year.round({'weighted_pct': 2, 'ci_low_pct': 2, 'ci_high_pct': 2, 'effective_n': 1}))

## 5. KGSS: 연도별 추세 검정

성공 요인별 중요 응답 여부가 조사연도 순서에 따라 증가·감소하는지 Cochran-Armitage trend test로 확인합니다.

해석상 주의: 검정은 유효 응답자 원자료의 비가중 빈도를 사용하고, 비율 표시는 KGSS 가중치를 적용합니다.

In [ ]:
trend_rows = []
for variable, label in success_labels.items():
    successes = []
    totals = []
    weighted_start = None
    weighted_end = None
    for year in success_years:
        valid = kgss[kgss['YEAR'].eq(year) & kgss[variable].isin(valid_values)].copy()
        important = valid[variable].isin(important_values)
        successes.append(int(important.sum()))
        totals.append(int(len(valid)))
        weighted_pct = kgss_success_ci_by_year.loc[
            (kgss_success_ci_by_year['YEAR'].eq(year)) & (kgss_success_ci_by_year['variable'].eq(variable)),
            'weighted_pct',
        ].iloc[0]
        if year == success_years[0]:
            weighted_start = weighted_pct
        if year == success_years[-1]:
            weighted_end = weighted_pct
    z, chi2, p_value = cochran_armitage_trend_test(successes, totals, success_years)
    trend_rows.append({
        'variable': variable,
        'label': label,
        'test': 'Cochran-Armitage trend test',
        'start_year': success_years[0],
        'end_year': success_years[-1],
        'weighted_pct_start': weighted_start,
        'weighted_pct_end': weighted_end,
        'weighted_pct_change_pp': weighted_end - weighted_start,
        'valid_n_total': sum(totals),
        'z': z,
        'chi2_df1': chi2,
        'p_value': p_value,
        'interpretation': '증가 추세' if z > 0 and p_value < 0.05 else ('감소 추세' if z < 0 and p_value < 0.05 else '뚜렷한 선형 추세 없음'),
    })

kgss_year_trend_tests = pd.DataFrame(trend_rows)
kgss_year_trend_tests['p_value_display'] = kgss_year_trend_tests['p_value'].apply(format_p_value)
trend_path = table_dir / '05_kgss_year_trend_tests.csv'
kgss_year_trend_tests.to_csv(trend_path, index=False, encoding='utf-8-sig')
print('저장:', trend_path)
display(kgss_year_trend_tests.round({'weighted_pct_start': 2, 'weighted_pct_end': 2, 'weighted_pct_change_pp': 2, 'z': 2, 'chi2_df1': 2, 'p_value': 4}))

## 6. KGSS: 2025년 연령대별 차이 검정

2025년 응답에서 연령대별 중요 응답 비율 차이가 있는지 카이제곱 독립성 검정과 Cramer's V로 확인합니다.

In [ ]:
age_2025 = kgss[kgss['YEAR'].eq(2025)].copy()
age_2025 = age_2025[age_2025['AGE'].between(20, 120)]
age_2025['age_group'] = pd.cut(
    age_2025['AGE'],
    bins=[20, 30, 40, 50, 60, np.inf],
    labels=['20대', '30대', '40대', '50대', '60대 이상'],
    right=False,
)

age_test_rows = []
for variable, label in success_labels.items():
    valid = age_2025[age_2025[variable].isin(valid_values) & age_2025['age_group'].notna()].copy()
    valid['important'] = valid[variable].isin(important_values).astype(int)
    table = pd.crosstab(valid['age_group'], valid['important']).reindex(index=['20대', '30대', '40대', '50대', '60대 이상'], columns=[0, 1], fill_value=0)
    chi2, df_chi, p_value, cramers_v = chi_square_independence(table.values)
    age_test_rows.append({
        'variable': variable,
        'label': label,
        'test': 'chi-square independence: age_group x important',
        'valid_n': int(table.values.sum()),
        'chi2': chi2,
        'df': df_chi,
        'p_value': p_value,
        'cramers_v': cramers_v,
        'interpretation': '연령대별 차이 확인' if p_value < 0.05 else '연령대별 차이 뚜렷하지 않음',
    })

kgss_age_group_tests_2025 = pd.DataFrame(age_test_rows)
kgss_age_group_tests_2025['p_value_display'] = kgss_age_group_tests_2025['p_value'].apply(format_p_value)
age_test_path = table_dir / '05_kgss_age_group_tests_2025.csv'
kgss_age_group_tests_2025.to_csv(age_test_path, index=False, encoding='utf-8-sig')
print('저장:', age_test_path)
display(kgss_age_group_tests_2025.round({'chi2': 2, 'p_value': 4, 'cramers_v': 3}))

## 7. 가계금융복지조사: 소득 분위와 순자산 분위 관련성 검정

소득과 순자산은 관련되어 있을 가능성이 높습니다. 검정 목적은 “무관하다”를 보이는 것이 아니라, 관련성의 크기와 불일치의 존재를 함께 보여주는 것입니다.

In [ ]:
hf_path = raw_dir / 'household_finance_2025' / '2025_가구마스터_20260604_10391.csv'
hf = pd.read_csv(hf_path, encoding='cp949')

income_col = '소득5분위코드(보완)'
net_asset_col = '순자산5분위코드'
weight_col = '가중값'
age_group_col = '가구주연령_10세단위코드'

quintile_map = {'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4, 'Q5': 5}
age_label_map = {
    'G1': '30세미만',
    'G2': '30~40세 미만',
    'G3': '40~50세 미만',
    'G4': '50~60세 미만',
    'G5': '60세이상',
}

hf_analysis = hf[[income_col, net_asset_col, weight_col, age_group_col]].copy()
hf_analysis['income_q'] = hf_analysis[income_col].map(quintile_map)
hf_analysis['net_asset_q'] = hf_analysis[net_asset_col].map(quintile_map)
hf_analysis['income_top40_net_asset_low40'] = hf_analysis[income_col].isin(['Q4', 'Q5']) & hf_analysis[net_asset_col].isin(['Q1', 'Q2'])
hf_analysis['income_low40_net_asset_top40'] = hf_analysis[income_col].isin(['Q1', 'Q2']) & hf_analysis[net_asset_col].isin(['Q4', 'Q5'])

valid_hf = hf_analysis.dropna(subset=['income_q', 'net_asset_q', weight_col]).copy()
print('가계금융복지조사 분석 행 수:', valid_hf.shape[0])

In [ ]:
unweighted_table = pd.crosstab(valid_hf['income_q'], valid_hf['net_asset_q']).reindex(index=[1,2,3,4,5], columns=[1,2,3,4,5], fill_value=0)
weighted_table = pd.crosstab(valid_hf['income_q'], valid_hf['net_asset_q'], values=valid_hf[weight_col], aggfunc='sum').reindex(index=[1,2,3,4,5], columns=[1,2,3,4,5], fill_value=0)

chi2, df_chi, p_value, cramers_v = chi_square_independence(unweighted_table.values)
weighted_spearman_like = weighted_corr(valid_hf['income_q'], valid_hf['net_asset_q'], valid_hf[weight_col])
unweighted_pearson_quintile = np.corrcoef(valid_hf['income_q'], valid_hf['net_asset_q'])[0, 1]

mismatch_age_rows = []
for metric_col, metric_label in [
    ('income_top40_net_asset_low40', '소득상위40_순자산하위40'),
    ('income_low40_net_asset_top40', '소득하위40_순자산상위40'),
]:
    table = pd.crosstab(valid_hf[age_group_col], valid_hf[metric_col].astype(int)).reindex(index=['G1','G2','G3','G4','G5'], columns=[0,1], fill_value=0)
    m_chi2, m_df, m_p, m_v = chi_square_independence(table.values)
    mismatch_age_rows.append({
        'metric': metric_label,
        'test': 'chi-square independence: age_group x mismatch_group',
        'valid_n': int(table.values.sum()),
        'chi2': m_chi2,
        'df': m_df,
        'p_value': m_p,
        'cramers_v': m_v,
        'interpretation': '연령대별 불일치 비율 차이 확인' if m_p < 0.05 else '연령대별 차이 뚜렷하지 않음',
    })

household_tests = pd.DataFrame([
    {
        'analysis': '소득5분위 × 순자산5분위',
        'test': 'chi-square independence',
        'valid_n': int(unweighted_table.values.sum()),
        'chi2': chi2,
        'df': df_chi,
        'p_value': p_value,
        'effect_size': 'Cramers V',
        'effect_value': cramers_v,
        'additional_metric': 'weighted ordinal correlation',
        'additional_value': weighted_spearman_like,
        'interpretation': '소득과 순자산은 관련되어 있지만 완전히 일치하지 않음',
    },
    {
        'analysis': '소득5분위 × 순자산5분위',
        'test': 'ordinal quintile correlation',
        'valid_n': int(unweighted_table.values.sum()),
        'chi2': np.nan,
        'df': np.nan,
        'p_value': np.nan,
        'effect_size': 'unweighted Pearson on quintile scores',
        'effect_value': unweighted_pearson_quintile,
        'additional_metric': 'weighted ordinal correlation',
        'additional_value': weighted_spearman_like,
        'interpretation': '분위 점수 기준 양의 관련성이 있으나 완전한 일치는 아님',
    },
])

household_age_tests = pd.DataFrame(mismatch_age_rows)
household_tests_full = pd.concat([household_tests, household_age_tests.rename(columns={'metric': 'analysis', 'cramers_v': 'effect_value'}).assign(effect_size='Cramers V', additional_metric='', additional_value=np.nan)], ignore_index=True, sort=False)

household_tests_full['p_value_display'] = household_tests_full['p_value'].apply(format_p_value)
household_test_path = table_dir / '05_household_income_net_asset_association_tests_2025.csv'
household_tests_full.to_csv(household_test_path, index=False, encoding='utf-8-sig')
print('저장:', household_test_path)
display(household_tests_full.round({'chi2': 2, 'p_value': 4, 'effect_value': 3, 'additional_value': 3}))

## 8. PPT용 검정 방법 요약표

In [ ]:
slide_method_summary = pd.DataFrame([
    {
        '슬라이드 목적': '성공 요인 인식 수준 제시',
        '사용 데이터': 'KGSS 2009, 2014, 2021, 2023, 2025',
        '분석 방법': '가중 중요 응답 비율 + Wilson 95% 신뢰구간(Kish effective n)',
        '검증/보강 포인트': '노력뿐 아니라 관계·배경 요인도 높은 수준으로 중요하게 인식되는지 확인',
        'PPT 해석 문장': '중요한 것은 1~2%p 차이가 아니라, 노력과 관계·배경 요인이 모두 높은 수준이라는 점이다.',
    },
    {
        '슬라이드 목적': '성공 요인 인식의 시간 변화 확인',
        '사용 데이터': 'KGSS 성공 요인 반복 문항',
        '분석 방법': 'Cochran-Armitage trend test',
        '검증/보강 포인트': '연도 순서에 따른 중요 응답 변화 방향 확인',
        'PPT 해석 문장': '노력의 중요성은 유지되며, 성공을 둘러싼 배경·관계 요인의 중요성도 함께 관찰된다.',
    },
    {
        '슬라이드 목적': '2025년 세대별 성공 조건 인식 차이 확인',
        '사용 데이터': 'KGSS 2025',
        '분석 방법': '연령대 × 중요 응답 여부 카이제곱 검정 + Cramer’s V',
        '검증/보강 포인트': '연령대별 차이가 있는 항목과 없는 항목 구분',
        'PPT 해석 문장': '세대 차이는 일부 항목에서 확인되지만, 성공 조건의 복합화라는 전체 흐름이 더 중요하다.',
    },
    {
        '슬라이드 목적': '현재 소득과 축적 자산의 관계 확인',
        '사용 데이터': '가계금융복지조사 2025 가구마스터',
        '분석 방법': '소득5분위 × 순자산5분위 교차표, 카이제곱 검정, Cramer’s V, 가중 분위상관',
        '검증/보강 포인트': '소득과 순자산은 관련되지만 완전히 일치하지 않음을 확인',
        'PPT 해석 문장': '순자산은 현재 소득만이 아니라 주택·축적 기간·생애주기 효과를 함께 반영한다.',
    },
])

method_path = table_dir / '05_slide_method_summary.csv'
slide_method_summary.to_csv(method_path, index=False, encoding='utf-8-sig')
print('저장:', method_path)
display(slide_method_summary)

## 9. PPT 후보 그림: 2025년 KGSS 성공 요인 신뢰구간

In [ ]:
plot_2025 = kgss_success_ci_by_year[kgss_success_ci_by_year['YEAR'].eq(2025)].copy()
plot_2025['yerr_low'] = plot_2025['weighted_pct'] - plot_2025['ci_low_pct']
plot_2025['yerr_high'] = plot_2025['ci_high_pct'] - plot_2025['weighted_pct']
plot_2025 = plot_2025.set_index('variable').loc[['SUCDEFRT', 'SUCDKNOW', 'SUCDWLTH', 'SUCDPAED']].reset_index()

fig, ax = plt.subplots(figsize=(8, 4.8))
colors = ['#2E86AB', '#3B7A57', '#A23B72', '#F18F01']
ax.bar(plot_2025['label'], plot_2025['weighted_pct'], color=colors, alpha=0.85)
ax.errorbar(
    plot_2025['label'],
    plot_2025['weighted_pct'],
    yerr=[plot_2025['yerr_low'], plot_2025['yerr_high']],
    fmt='none',
    ecolor='black',
    elinewidth=1.4,
    capsize=5,
)
for _, row in plot_2025.iterrows():
    ax.text(row['label'], row['weighted_pct'] + 1.0, f"{row['weighted_pct']:.1f}%", ha='center', va='bottom', fontsize=11)
ax.set_ylim(80, 101)
ax.set_ylabel('가중 중요 응답 비율(%)')
ax.set_title('2025년 성공 요인 중요 인식: 노력만이 아니라 관계·배경도 높다')
ax.grid(axis='y', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig_path = figure_dir / '05_kgss_success_2025_ci.png'
fig.savefig(fig_path, dpi=200, bbox_inches='tight')
print('저장:', fig_path)
plt.show()

## 10. 해석 원칙

- KGSS는 성공 요인에 대한 인식을 측정한다. 실제 성공의 원인을 직접 검증하는 자료가 아니다.
- 가계금융복지조사는 경제적 배경 자료다. 성공 인식 변화의 원인을 직접 설명하는 인과 자료가 아니다.
- p-value는 보조 정보다. 표본이 크면 작은 차이도 유의해질 수 있으므로, Cramer's V와 실제 비율 차이를 함께 본다.
- 핵심 결론은 “노력의 부정”이 아니라 “성공 공식의 복합화”다.